# ZoMBI reproducibility notebook

This notebook is the manuscript-facing entry point for the ZoMBI comparison used in the tidyHEBO repository.

It is intended to be run from the **tidyHEBO environment**. The notebook:

- loads the archived reference traces used for the ZoMBI comparison,
- rebuilds the **article-style two-panel figure** for the two retained tasks,
- keeps the plotting code in one place,
- and documents how to launch fresh benchmark runs when needed.

The article figure contains only **Poisson ratio** and **thermoelectric**. **Wildfire is excluded**.


## Expected location

Run this notebook either:

1. from inside the `04_needle_in_a_haystack/` directory, or
2. from the repository root that contains `04_needle_in_a_haystack/`.

No separate reproducibility-specific environment is required.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().resolve()
if not (ROOT / 'reference_results').is_dir():
    candidate = ROOT / '04_needle_in_a_haystack'
    if (candidate / 'reference_results').is_dir():
        ROOT = candidate
    else:
        raise FileNotFoundError(
            'Could not find reference_results/. Start the notebook from the '
            '04_needle_in_a_haystack directory or from the repository root.'
        )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.tasks import TASKS

RESULTS_ROOT = ROOT / 'reference_results'
BUDGET = 100
print(f'ZoMBI reproducibility root: {ROOT}')


## Reference methods and data loading

The colors and line styles follow the manuscript convention:

- solid lines: tidyHEBO / LogEI / HEBO / Optuna / Random,
- dashed lines: ZoMBI variants.


In [ ]:
METHODS = [
    ('tidyhebo', 'tidyHEBO', 'tab:red', '-'),
    ('logei', 'LogEI', 'tab:orange', '-'),
    ('hebo', 'HEBO', 'tab:purple', '-'),
    ('optuna', 'optuna', 'tab:green', '-'),
    ('random', 'random', 'tab:blue', '-'),
    ('zombi_ada', 'ZoMBI (LCB Adaptive)', 'tab:blue', '--'),
    ('zombi_abrupt', 'ZoMBI (EI Abrupt)', 'tab:orange', '--'),
    ('zombi_EI', 'ZoMBI (EI)', 'tab:red', '--'),
    ('zombi_LCB', 'ZoMBI (LCB)', 'tab:green', '--'),
]

NEEDLE_TEXT = {
    'poisson_ratio': [
        ('Needle 1', -1.2, 2.0, 0.04),
        ('Needle 2', -1.7, 2.0, 0.04),
    ],
    'thermoelectric': [
        ('Needle 1', 1.4, 40.0, -0.03),
    ],
}

PANEL_SPECS = [
    ('poisson_ratio', 'A'),
    ('thermoelectric', 'B'),
]


def cumulative_min(values):
    return np.minimum.accumulate(np.asarray(values, dtype=float), axis=1)


def load_trace(task_name, method, budget=BUDGET):
    task_dir = RESULTS_ROOT / task_name
    if method == 'random':
        path = task_dir / 'random_values_raw.txt'
        if not path.exists():
            path = task_dir / 'random_values.txt'
        values = np.loadtxt(path)
        if values.ndim == 1:
            if values.size % budget:
                raise ValueError(f'Cannot reshape {path} into runs x {budget}')
            values = values.reshape(-1, budget)
    else:
        path = task_dir / f'{method}_values.txt'
        if not path.exists():
            path = task_dir / f'{method}.txt'
        values = np.loadtxt(path, ndmin=2)

    values = cumulative_min(values)
    return values[:, -budget:]


def available_run_counts(task_name):
    counts = {}
    for method, label, _, _ in METHODS:
        try:
            counts[label] = load_trace(task_name, method).shape[0]
        except (OSError, ValueError):
            continue
    return counts


## Article figure builder

The function below rebuilds the two-panel figure in the article style:


In [ ]:
def _plot_panel(ax, task_name, panel_letter, budget=BUDGET):
    task = TASKS[task_name]
    x = np.arange(1, budget + 1)
    handles = []
    labels = []

    plotted = 0
    for method, label, color, linestyle in METHODS:
        try:
            raw = load_trace(task_name, method, budget)
        except (OSError, ValueError):
            continue

        displayed = task.display(raw)
        median = np.median(displayed, axis=0)
        line, = ax.plot(
            x,
            median,
            label=label,
            color=color,
            linestyle=linestyle,
            linewidth=2,
        )
        handles.append(line)
        labels.append(label)
        plotted += 1

    if plotted == 0:
        raise FileNotFoundError(f'No result traces found for task: {task_name}')

    for needle_label, level, x_text, y_offset in NEEDLE_TEXT[task_name]:
        ax.axhline(level, color='black', linestyle='--', linewidth=1.1, alpha=0.8)
        ax.text(
            x_text,
            level + y_offset,
            needle_label,
            fontsize=9,
            ha='left',
            va='bottom' if y_offset >= 0 else 'top',
        )

    ylabel = 'best value'
    ax.set_xlabel('number of function evaluations')
    ax.set_ylabel(ylabel)
    ax.set_xlim(0, budget)
    ax.grid(alpha=0.35)

    ax.text(
        -0.12,
        1.00,
        panel_letter,
        transform=ax.transAxes,
        fontsize=19,
        fontweight='bold',
        ha='left',
        va='top',
        clip_on=False,
    )

    return handles, labels


def plot_manuscript_figure(budget=BUDGET):
    fig, axes = plt.subplots(1, 2, figsize=(12.2, 4.8))

    handles = labels = None
    for ax, (task_name, panel_letter) in zip(axes, PANEL_SPECS):
        panel_handles, panel_labels = _plot_panel(ax, task_name, panel_letter, budget)
        if handles is None:
            handles, labels = panel_handles, panel_labels

    legend = fig.legend(
        handles,
        labels,
        loc='lower center',
        bbox_to_anchor=(0.5, -0.02),
        ncol=3,
        frameon=False,
        fontsize=10,
    )

    fig.tight_layout(rect=(0.02, 0.16, 1.0, 1.0))
    return fig, axes, legend


## Rebuild the manuscript figure

Running the next cell should display the final article-style figure directly in the notebook.


In [ ]:
manuscript_fig, manuscript_axes, manuscript_legend = plot_manuscript_figure()
if 'agg' not in plt.get_backend().lower():
    plt.show()


## Available archived runs

The next cell prints how many runs are available per method for each retained task.


In [ ]:
for task_name, _ in PANEL_SPECS:
    print(task_name)
    for label, count in available_run_counts(task_name).items():
        print(f'  {label:<24} {count}')
    print()


## Fresh benchmark runs (optional)

The article figure above uses the archived reference traces under `reference_results/`.

To launch a fresh run from the tidyHEBO environment, use `run_benchmark.py`. These commands are **examples only** and are **not executed automatically** by this notebook.


In [ ]:
fresh_run_examples = r'''
# thermoelectric smoke run
python run_benchmark.py   --task thermoelectric   --optimizer tidyhebo   --zombi-root /path/to/ZoMBI   --runs 1   --evaluations 10   --seed 42   --output-dir smoke_results

# poisson_ratio smoke run
python run_benchmark.py   --task poisson_ratio   --optimizer tidyhebo   --zombi-root /path/to/ZoMBI   --runs 1   --evaluations 10   --seed 42   --output-dir smoke_results
'''
print(fresh_run_examples)
